In [ ]:
import pandas as pd

file_path = 'dataset.xlsx'
# if dataset is excel
df = pd.read_excel(file_path)
# if dataset is csv
#df = pd.read_csv(file_path, sep=';', on_bad_lines='skip', encoding='utf-8', engine='python')

# dropping unnecessary columns
df = df.drop(columns=['Column11', 'Column12'])
# dropping NA from target variable
df = df.dropna(subset=['credit_indicator'])
print("Dataset shape:", df.shape)

In [ ]:
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# ENCODING CITIES TO REGIONS
city_to_region = {
    'Petropavl': 'North',
    'Taraz': 'South',
    'Aktau': 'West',
    'Oskemen': 'East',
    'Almaty': 'Almaty',
    'Astana': 'Astana'
}

df_encoded = df.copy()

df_encoded['City'] = df_encoded['City'].map(city_to_region)

print("Value counts for 'City' after mapping:")
print(df_encoded['City'].value_counts(dropna=False))

print("Value counts for 'Category':")
print(df_encoded['Category'].value_counts(dropna=False))

# DUMMIES
df_encoded = pd.get_dummies(df_encoded, columns=['City'], prefix='City', dtype=int)
df_encoded = df_encoded.drop(columns=['City_Almaty'])

df_encoded = pd.get_dummies(df_encoded, columns=['Category'], prefix='Category', dtype=int)
df_encoded = df_encoded.drop(columns=['Category_Accessories'])

df_encoded = pd.get_dummies(df_encoded, columns=['Counterparty'], prefix='Counterparty', dtype=int)
df_encoded = df_encoded.drop(columns=['Counterparty_Individual Entrepreneur'])

# FORMATTING 'PERIOD' COLUMN
df_encoded['Period'] = pd.to_datetime(df_encoded['Period'], format='%d.%m.%Y %H:%M:%S')

# ENGINEERING WEEKEND INDICATOR FROM 'PERIOD'
df_encoded['is_weekend'] = df_encoded['Period'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

# ENGINEERING HOUR INDICATOR FROM 'PERIOD'
df_encoded['purchase_hour'] = df_encoded['Period'].dt.hour
df_encoded = df_encoded[(df_encoded['purchase_hour'] != 0)]

df_encoded = df_encoded.drop(columns=['Period'])

df_encoded['Sale'] = df_encoded['Sale'].astype(str).str.replace(' ', '', regex=False)
#df_encoded['ProdNumber'] = df_encoded['ProdNumber'].astype(str).str.replace(' ', '', regex=False)

df_encoded['Sale'] = pd.to_numeric(df_encoded['Sale'], errors='coerce')
#df_encoded['ProdNumber'] = pd.to_numeric(df_encoded['ProdNumber'], errors='coerce')

df_encoded.dropna(subset=['Sale', 'ProdNumber'], inplace=True)

df_encoded = df_encoded[df_encoded['ProdNumber'] != 0].copy() # Added .copy() here

# ENGINEERING PRICE INDICATOR
df_encoded['Price'] = df_encoded['Sale'] / (df_encoded['ProdNumber'])
df_encoded['Price'] = round(df_encoded['Price']).astype(int)

# df_encoded = df_encoded.drop(columns=['Sale', 'ProdNumber'])
#print(df_encoded.head(5))

# DROPPING EVERYTHING UNDER 1000 SINCE THEY'RE NOT RELEVANT
df_encoded = df_encoded[(df_encoded['Price'] > 1000)]
df_encoded = df_encoded[(df_encoded['Sale'] > 1000)]

# DROPPING COLUMNS WE NO LONGER NEED
df_encoded = df_encoded.drop(columns=['Sale', 'ProdNumber'])
print("Dataset shape:", df_encoded.shape)
print("Data is encoded")
display(df_encoded.head())



target = 'credit_indicator'
X = df_encoded.drop(columns=[target])
y = df_encoded[target]

# SPLITTING DATA
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

zero_variance_cols = X_train.columns[X_train.var() == 0]
if not zero_variance_cols.empty:
    print(f"Dropping zero-variance columns from X_train: {list(zero_variance_cols)}")
    X_train_clean = X_train.drop(columns=zero_variance_cols)
else:
    X_train_clean = X_train.copy()

numerical_cols = ['Price', 'purchase_hour']

numerical_cols_to_scale = [col for col in numerical_cols if col in X_train_clean.columns]

# STANDARD SCALER
if numerical_cols_to_scale:
    scaler = StandardScaler()
    X_train_clean[numerical_cols_to_scale] = scaler.fit_transform(X_train_clean[numerical_cols_to_scale])

X_train_sm = sm.add_constant(X_train_clean)

logit_model = sm.Logit(y_train, X_train_sm)
result = logit_model.fit_regularized(method='l1', alpha=0.1, maxiter=1000)

print(result.summary())

In [ ]:
# TRAINING DATA, ACCURACY AND CONFUSION MATRIX

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

target = 'credit_indicator'
X = df_encoded.drop(columns=[target])
y = df_encoded[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

# LOGISTIC REGRESSION
model = LogisticRegression(max_iter=1000, solver='liblinear', class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:\n", cm)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", report)

In [ ]:
# FOR ROBUSTNESS CHECK - PROBIT MODEL

import statsmodels.api as sm

probit_model = sm.Probit(y_train, X_train_sm)
probit_result = probit_model.fit(disp=False)

print("\nProbit Regression Results:")
print(probit_result.summary())

In [ ]:
# DECRIPTIVE STATISTICS

import pandas as pd

descriptive_stats = df_encoded.describe().transpose()

stats_table = descriptive_stats[['mean', 'std', 'min', 'max']]
stats_table.columns = ['Mean', 'Standard Deviation', 'Min', 'Max']

print("Descriptive Statistics for Variables:")
display(stats_table)

In [ ]:
# VIF - to check for multicollinearity

import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

if 'const' not in X_train_clean.columns:
    X_train_sm_for_vif = sm.add_constant(X_train_clean)
else:
    X_train_sm_for_vif = X_train_clean.copy()

vif_data = pd.DataFrame()
vif_data["feature"] = X_train_sm_for_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_train_sm_for_vif.values, i) for i in range(X_train_sm_for_vif.shape[1])]

vif_data = vif_data.sort_values(by="VIF", ascending=False)

display(vif_data)

In [ ]:
# PIECHART FOR CITY (did include in thesis)

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

sizes = np.array([14138, 12431, 6538, 5951, 3820])
labels = ["North", "Astana", "East", "Center", "Almaty"]

plt.figure(figsize=(8, 8))
city_counts = df_encoded['City'].value_counts()
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Samples by Region')
plt.axis('equal')
plt.show()

In [ ]:
# PIECHART FOR CATEGORY (didn't include)
plt.figure(figsize=(10, 10))
category_counts = df['Category'].value_counts()
plt.pie(uroven2_counts, labels=uroven2_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Samples by Category')
plt.axis('equal')
plt.show()

In [ ]:
# PIECHART FOR COUNTERPARTY (didn't include)
plt.figure(figsize=(8, 8))
counterparty_counts = df['Counterparty'].value_counts()
plt.pie(kontragent_counts, labels=kontragent_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Samples by Counterparty')
plt.axis('equal')
plt.show()

In [ ]:
# remnants of the time when I tried Random Forest for prediction (it actually showed higher accuracy than LR, however I decided not to include it to the thesis)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    class_weight = 'balanced',
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

importances = model.feature_importances_

accuracy = accuracy_score(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print("\nClassification Report:\n", classification_rep)
print(f"\nFeature importances: {importances}")